# Omnitech.SignalCortex — Exploratory Data Analysis

Analyzes `market_data_features` to understand the data before modeling.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from configs.config import load_config
from data.db import DatabaseConnection

sns.set_theme(style='whitegrid')
%matplotlib inline

config = load_config('../configs/default.yaml')
print('Config loaded')

## 1. Data Loading and Overview

In [ ]:
with DatabaseConnection(config.database) as db:
    df = db.fetch_features(
        pair_name=config.data.pair_name,
        timeframe=config.data.timeframe,
        feature_columns=config.data.feature_columns,
        label_column=config.data.label_column,
    )

print(f'Shape: {df.shape}')
print(f'Period: {df.candle_open_time.min()} to {df.candle_open_time.max()}')
print(f'Columns: {list(df.columns)}')
df.dtypes

In [ ]:
df.describe()

## 2. Label Distribution

In [ ]:
label_counts = df['buy_signal'].value_counts()
print('Label distribution:')
print(label_counts)
print(f'BUY ratio: {label_counts.get(True, 0) / len(df):.2%}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
label_counts.plot.bar(ax=axes[0], title='BUY vs HOLD counts')
axes[0].set_xticklabels(['HOLD (False)', 'BUY (True)'], rotation=0)
label_counts.plot.pie(ax=axes[1], autopct='%1.1f%%', labels=['HOLD', 'BUY'])
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()

## 3. Temporal Label Distribution

In [ ]:
df_time = df.set_index('candle_open_time')
monthly_buy_rate = df_time['buy_signal'].resample('ME').mean()

fig, ax = plt.subplots(figsize=(14, 4))
monthly_buy_rate.plot(ax=ax, marker='o')
ax.set_title('Monthly BUY signal rate over time')
ax.set_ylabel('BUY rate')
ax.axhline(monthly_buy_rate.mean(), color='red', linestyle='--', label='Mean')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Feature Correlation Heatmap

In [ ]:
feature_df = df[config.data.feature_columns]
corr = feature_df.corr()

fig, ax = plt.subplots(figsize=(18, 16))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap='coolwarm', center=0,
            linewidths=0.5, ax=ax)
ax.set_title('Feature correlation matrix (lower triangle)')
plt.tight_layout()
plt.show()

# Highly correlated pairs (|corr| > 0.9)
high_corr = []
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        c = corr.iloc[i, j]
        if abs(c) > 0.9:
            high_corr.append((corr.columns[i], corr.columns[j], c))
print(f'Highly correlated pairs (|r|>0.9): {len(high_corr)}')
for a, b, r in sorted(high_corr, key=lambda x: -abs(x[2])):
    print(f'  {a} vs {b}: {r:.3f}')

## 5. Feature Distributions

In [ ]:
n_features = len(config.data.feature_columns)
n_cols = 4
n_rows = (n_features + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 3))
axes = axes.flatten()

for i, col in enumerate(config.data.feature_columns):
    axes[i].hist(df[col].dropna(), bins=50, edgecolor='none', alpha=0.7)
    axes[i].set_title(col, fontsize=9)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature distributions', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Preliminary Feature Importance (Random Forest)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

X = df[config.data.feature_columns].fillna(0)
y = df['buy_signal'].astype(int)

# Use only 30% of data for speed
sample_size = min(50000, len(X))
idx = np.random.choice(len(X), size=sample_size, replace=False)
X_sample, y_sample = X.iloc[idx], y.iloc[idx]

rf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
rf.fit(X_sample, y_sample)

importances = pd.Series(rf.feature_importances_, index=config.data.feature_columns)
importances = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 12))
importances.plot.barh(ax=ax)
ax.set_title('Random Forest feature importance')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

print('Top 10 features:')
print(importances.tail(10))

## 7. S/R Feature Analysis

In [ ]:
sr_features = ['dist_support_pct', 'dist_resistance_pct', 'sr_zone_position']
available = [f for f in sr_features if f in df.columns]

fig, axes = plt.subplots(1, len(available), figsize=(6 * len(available), 4))
if len(available) == 1:
    axes = [axes]

for ax, col in zip(axes, available):
    for label, group in df.groupby('buy_signal'):
        group[col].hist(bins=40, alpha=0.6, ax=ax, label='BUY' if label else 'HOLD')
    ax.set_title(f'{col} by label')
    ax.legend()

plt.tight_layout()
plt.show()

## 8. Stationarity Check (ADF Test)

In [ ]:
from statsmodels.tsa.stattools import adfuller

results = []
for col in config.data.feature_columns:
    series = df[col].dropna()
    if len(series) < 100:
        continue
    try:
        adf_result = adfuller(series[:5000], autolag='AIC')  # limit for speed
        results.append({
            'feature': col,
            'adf_stat': adf_result[0],
            'p_value': adf_result[1],
            'stationary': adf_result[1] < 0.05
        })
    except Exception as e:
        results.append({'feature': col, 'adf_stat': None, 'p_value': None, 'stationary': None})

adf_df = pd.DataFrame(results)
print(f'Non-stationary features (p > 0.05):')
non_stationary = adf_df[adf_df['stationary'] == False]
print(non_stationary[['feature', 'p_value']].to_string(index=False))
print(f'\nStationary: {(adf_df.stationary == True).sum()} / {len(adf_df)}')

## 9. Class Separability

In [ ]:
# Top 6 features by RF importance
top_features = importances.tail(6).index.tolist()
n = len(top_features)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    for label, group in df.groupby('buy_signal'):
        axes[i].hist(group[feat].dropna(), bins=40, alpha=0.5,
                     label='BUY' if label else 'HOLD', density=True)
    axes[i].set_title(feat)
    axes[i].legend(fontsize=8)

plt.suptitle('Top features: BUY vs HOLD distribution', fontsize=13)
plt.tight_layout()
plt.show()

## 10. Recommendations

Based on the analysis above:

- **High-correlation pairs**: Consider removing redundant features (e.g., price-level EMAs when ratio features exist).
- **Non-stationary features**: These may need differencing or ratio normalization. The model should handle this via BatchNorm + RobustScaler, but watch for train/test distribution shift.
- **Class imbalance**: `auto_class_weights=true` in config compensates. Monitor BUY precision > 60% to avoid too many false BUY signals.
- **Recommended initial feature set**: Prioritize top 20 by RF importance and bounded/ratio features (RSI, StochRSI, bb_pctb, price_ema ratios, sr_zone_position) which are already scale-invariant.